In [23]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [24]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

In [25]:
class Blog(TypedDict):

    title : str
    outline : str
    content : str
    evaluate : int

In [26]:
def create_outline(state: Blog) -> Blog:

    title = state['title']
    prompt=f'generate a detailed outline on the topic - {title}'
    outline=model.invoke(prompt).content

    state['outline']=outline

    return state

In [27]:
def create_content(state : Blog) -> Blog:

    title=state['title']
    outline=state['outline']
    prompt=f'create a detailed blog on the topic-{title} along with given outline-{outline}'

    content=model.invoke(prompt).content

    state['content']=content

    return state

In [28]:
def evaluate(state : Blog) -> Blog:

    title=state['title']
    outline=state['outline']
    content=state['content']

    prompt=f'based on given title->{title} and outline->{outline} and content->{content} evaluate the content on the scale of 1 to 10'

    evaluate=model.invoke(prompt).content

    state['evaluate']=evaluate

    return state

In [29]:
graph = StateGraph(Blog)

graph.add_node('create_outline',create_outline)
graph.add_node('create_content',create_content)
graph.add_node('evaluate',evaluate)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_content')
graph.add_edge('create_content','evaluate')
graph.add_edge('evaluate',END)

workflow=graph.compile()

In [30]:
initial_state={'title':'corona virus'}

final_state=workflow.invoke(initial_state)



In [31]:
print(final_state['outline'])
print(final_state['content'])


[{'type': 'text', 'text': 'This outline provides a comprehensive overview of the COVID-19 pandemic, covering its scientific origins, global impact, and long-term consequences.\n\n---\n\n### **I. Introduction**\n*   **Definition:** What is SARS-CoV-2 and COVID-19?\n*   **The Global Timeline:** Brief overview of the initial outbreak in late 2019 (Wuhan, China) to the declaration of a global pandemic by the WHO.\n*   **Significance:** The unprecedented scale of socio-economic and public health disruption.\n\n### **II. Virology and Biology of the Virus**\n*   **Structure:** Explanation of the "Crown" (corona) spikes and how they attach to human cells (ACE2 receptors).\n*   **Transmission Mechanisms:**\n    *   Respiratory droplets and aerosols.\n    *   Surface transmission (fomites).\n    *   The role of asymptomatic and pre-symptomatic spread.\n*   **Variants and Mutation:** How viruses evolve (e.g., Alpha, Delta, Omicron) and the implications for transmissibility and immune evasion.\n\n

In [ ]:
print(final_state['evaluate'])

AttributeError: 'list' object has no attribute 'content'